# Internet Payment Tracking Colab Starter

This notebook supports two data sources:

- Google Drive zip exports from Django
- Firebase Firestore collections synced from Django

It gives you a baseline analysis of face enrollment data, payments, and overdue state.


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn firebase-admin pillow

In [ ]:
import base64
import json
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.ensemble import IsolationForest

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)


In [ ]:
try:
    from google.colab import drive, userdata  # type: ignore
except Exception:
    drive = None
    userdata = None


def secret(name, default=''):
    value = os.environ.get(name)
    if value:
        return value
    if userdata is not None:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass
    return default

if drive is not None:
    try:
        drive.mount('/content/drive', force_remount=False)
    except Exception as exc:
        print(f'Drive mount skipped: {exc}')

SOURCE_MODE = secret('COLAB_SOURCE_MODE', 'drive').strip().lower()
DRIVE_EXPORT_DIR = Path(secret('COLAB_DRIVE_EXPORT_DIR', '/content/drive/MyDrive/colab_exports'))
DRIVE_OUTPUT_DIR = Path(secret('COLAB_DRIVE_OUTPUT_DIR', '/content/drive/MyDrive/colab_output'))
DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

firebase_db = None
if SOURCE_MODE == 'firebase':
    import firebase_admin
    from firebase_admin import credentials, firestore

    firebase_creds = secret('FIREBASE_CREDENTIALS_PATH', '')
    if not firebase_creds:
        raise ValueError('Set FIREBASE_CREDENTIALS_PATH to your Firebase service account JSON path.')
    if not firebase_admin._apps:
        firebase_admin.initialize_app(credentials.Certificate(firebase_creds))
    firebase_db = firestore.client()


def latest_zip_in(directory):
    directory = Path(directory)
    if not directory.exists():
        return None
    zips = sorted(directory.glob('*.zip'), key=lambda item: item.stat().st_mtime, reverse=True)
    return zips[0] if zips else None

latest_bundle = latest_zip_in(DRIVE_EXPORT_DIR)
BUNDLE_PATH = secret('COLAB_BUNDLE_PATH', str(latest_bundle) if latest_bundle else str(DRIVE_EXPORT_DIR / 'colab_bundle.zip'))
OUTPUT_DIR = Path(secret('COLAB_OUTPUT_DIR', str(DRIVE_OUTPUT_DIR)))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Source mode:', SOURCE_MODE)
print('Bundle path:', BUNDLE_PATH)
print('Output dir:', OUTPUT_DIR)
if latest_bundle:
    print('Latest Drive bundle:', latest_bundle)


In [ ]:
def load_bundle(bundle_path):
    bundle_path = Path(bundle_path)
    if bundle_path.is_dir():
        return bundle_path
    if not bundle_path.exists():
        raise FileNotFoundError(f'Bundle not found: {bundle_path}')
    extract_dir = OUTPUT_DIR / bundle_path.stem
    extract_dir.mkdir(parents=True, exist_ok=True)
    if bundle_path.suffix.lower() == '.zip':
        with zipfile.ZipFile(bundle_path, 'r') as archive:
            archive.extractall(extract_dir)
        return extract_dir
    return bundle_path.parent


def firestore_collection_df(collection_name):
    docs = firebase_db.collection(collection_name).stream()
    records = []
    for doc in docs:
        payload = doc.to_dict() or {}
        payload['_doc_id'] = doc.id
        records.append(payload)
    return pd.DataFrame(records)


def load_data_frame(source, drive_path=None, firestore_collection=None):
    if source == 'firebase':
        if not firestore_collection:
            return pd.DataFrame()
        return firestore_collection_df(firestore_collection)
    if drive_path is None:
        return pd.DataFrame()
    return pd.read_csv(drive_path)


def load_json_or_empty(path):
    path = Path(path)
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding='utf-8'))


In [ ]:
if SOURCE_MODE == 'firebase':
    summary = {}
    clients_df = load_data_frame('firebase', firestore_collection='clients')
    payments_df = load_data_frame('firebase', firestore_collection='payments')
    overdue_df = load_data_frame('firebase', firestore_collection='overdue_snapshot')
    facial_records = load_data_frame('firebase', firestore_collection='facial_biometrics')
    face_manifest = pd.DataFrame()
else:
    bundle_dir = load_bundle(BUNDLE_PATH)
    print('Loaded bundle dir:', bundle_dir)
    print('Files:', sorted([p.name for p in bundle_dir.iterdir()]))
    summary = load_json_or_empty(bundle_dir / 'summary.json')
    clients_df = load_data_frame('drive', bundle_dir / 'clients.csv')
    payments_df = load_data_frame('drive', bundle_dir / 'payments.csv')
    overdue_df = load_data_frame('drive', bundle_dir / 'overdue_snapshot.csv')
    face_manifest = load_data_frame('drive', bundle_dir / 'face_images_manifest.csv')
    facial_path = bundle_dir / 'facial_biometrics.jsonl'
    facial_records = pd.read_json(facial_path, lines=True) if facial_path.exists() else pd.DataFrame()

summary, clients_df.head(), payments_df.head(), overdue_df.head(), facial_records.head()


## Face Template Analysis

This section reviews enrolled biometric templates and their quality metrics. If you later switch to storing more image data in Firebase, the notebook can use that too.


In [ ]:
if facial_records.empty:
    print('No facial biometrics found in this source.')
else:
    face_df = facial_records.copy()
    if 'template' in face_df.columns:
        face_df['descriptor_count'] = face_df['template'].apply(lambda x: len((x or {}).get('descriptors', [])) if isinstance(x, dict) else 0)
        face_df['template_method'] = face_df['template'].apply(lambda x: (x or {}).get('method', 'unknown') if isinstance(x, dict) else 'unknown')
    else:
        face_df['descriptor_count'] = 0
        face_df['template_method'] = 'unknown'
    display(face_df[['username', 'email', 'quality_score', 'confidence', 'descriptor_count', 'template_method']].head(20))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    sns.histplot(face_df['quality_score'], bins=10, ax=axes[0], color='#7c3aed')
    axes[0].set_title('Face Quality')
    sns.histplot(face_df['confidence'], bins=10, ax=axes[1], color='#0ea5e9')
    axes[1].set_title('Enrollment Confidence')
    sns.histplot(face_df['descriptor_count'], bins=10, ax=axes[2], color='#10b981')
    axes[2].set_title('Descriptor Count')
    plt.tight_layout()

    face_features = face_df[['quality_score', 'confidence', 'descriptor_count']].fillna(0)
    if len(face_features) >= 5:
        face_anomaly = IsolationForest(random_state=42, contamination=min(0.2, max(0.05, 1.0 / len(face_features))))
        face_df['anomaly_score'] = face_anomaly.fit_predict(face_features)
        display(face_df.sort_values(['anomaly_score', 'quality_score'])[['username', 'quality_score', 'confidence', 'descriptor_count', 'anomaly_score']].head(20))
    else:
        print('Not enough face records for anomaly scoring.')


## Payment and Overdue Analysis

The billing side remains rule-based in Django, but this view helps you inspect the data that Colab is reading.


In [ ]:
clients = clients_df.copy()
if not clients.empty:
    if 'due_date' in clients.columns:
        clients['due_date'] = pd.to_datetime(clients['due_date'], errors='coerce')
    if 'joined' in clients.columns:
        clients['joined'] = pd.to_datetime(clients['joined'], errors='coerce')
    clients['balance'] = pd.to_numeric(clients.get('balance', 0), errors='coerce').fillna(0)
    clients['fee'] = pd.to_numeric(clients.get('fee', 0), errors='coerce').fillna(0)
    clients['days_overdue'] = pd.to_numeric(clients.get('days_overdue', 0), errors='coerce').fillna(0)
    clients['is_overdue'] = clients['status'].isin(['Overdue', 'Disconnected']).astype(int)

    display(clients[['client_id', 'name', 'status', 'balance', 'days_overdue', 'plan']].sort_values('days_overdue', ascending=False).head(20))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.countplot(data=clients, x='status', ax=axes[0], palette='Set2')
    axes[0].set_title('Client status')
    sns.countplot(data=payments_df, x='status', ax=axes[1], palette='Set3')
    axes[1].set_title('Payment status')
    plt.tight_layout()
else:
    print('No client data available in this source.')


## Next Step

If you want stronger face results, switch the source mode to Firebase only after syncing the `facial_biometrics` collection with `sample_image_b64` values, or keep using the Drive bundle for raw image training.
